# 1 — Quick start: QC, density, and spatial statistics

This notebook runs end to end on synthetic data, so it needs no download
and is executed in CI on every push.

Substitute `synthetic_tissue()` with your own `AnnData` — SpatioEv only
requires `X_centroid`, `Y_centroid` and `imageid` in `.obs`.

## A synthetic tissue

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad

rng = np.random.default_rng(0)

def synthetic_tissue(n_per_niche=120, n_niches=3, seed=0):
    """Three spatially separated cell niches with two marker features."""
    rng = np.random.default_rng(seed)
    centres = np.array([[0.0, 0.0], [600.0, 0.0], [0.0, 600.0]])[:n_niches]
    coords, niche = [], []
    for i, c in enumerate(centres):
        coords.append(c + rng.normal(0, 60, (n_per_niche, 2)))
        niche += [f'niche_{i}'] * n_per_niche
    coords = np.vstack(coords)
    n = len(coords)
    obs = pd.DataFrame({
        'X_centroid': coords[:, 0],
        'Y_centroid': coords[:, 1],
        'imageid': 'demo_image',
        'label': np.arange(n),
        'niche': niche,
        'phenotype': rng.choice(['duct', 'immune', 'stroma'], n, p=[.4, .35, .25]),
        'area': rng.lognormal(4.6, 0.3, n),
        'nc_ratio': rng.uniform(0.15, 0.75, n),
        # a feature with real spatial structure, for the statistics below
        'gradient': coords[:, 0] + coords[:, 1] + rng.normal(0, 30, n),
        'noise': rng.normal(size=n),
    })
    adata = ad.AnnData(X=rng.lognormal(0, 1, (n, 2)))
    adata.var_names = ['CD8', 'Ki67']
    adata.obs = obs
    adata.obs_names = [f'cell_{i}' for i in range(n)]
    return adata

adata = synthetic_tissue()
adata

## Segmentation QC

`run_segmentation_qc` converts pixel measurements to physical units and
flags implausible objects.

In [ ]:
import spatioev as sv
from spatioev.config import QCConfig

qc = sv.pp.run_segmentation_qc(adata.copy(), QCConfig(pixel_size=0.325))
sv.pp.generate_qc_summary(qc, groupby='imageid')

## Tile density

`assign_tiles` bins cells onto a grid; the density functions then
summarise counts and occupied area per tile.

In [ ]:
tiles = sv.tl.assign_tiles(adata, tile_size=100)
density = sv.tl.compute_general_density(tiles, tile_size=100)
density.head()

In [ ]:
by_phenotype = sv.tl.compute_phenotype_density(
    tiles, phenotype_key='phenotype', tile_size=100
)
by_phenotype.head()

## Spatial autocorrelation

Moran's I asks whether similar values sit next to each other.
`gradient` was built with spatial structure; `noise` was not.

In [ ]:
coords = adata.obs[['X_centroid', 'Y_centroid']].to_numpy()

for feature in ['gradient', 'noise']:
    i = sv.tl.morans_i(coords, adata.obs[feature], k=8)
    print(f'{feature:>9}: Moran I = {i: .4f}')

A permutation test puts that on an inferential footing. The spatial
weight matrix is built once and reused across simulations, so 999
permutations is inexpensive.

In [ ]:
res = sv.tl.morans_i_permutation_test(
    coords, adata.obs['gradient'].to_numpy(), k=8, n_sim=999, random_state=0
)
{k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()}

## Ripley's K

`L - r` is positive for clustering and near zero under complete spatial
randomness. The synthetic tissue is deliberately clustered.

In [ ]:
sv.tl.ripleys_k(coords, radius=100.0)